### Relevant Python 3.9 Changes

The release of Python 3.9 has brought some new features.

This is a summary of the ones _I_ deemed relevant to this course, and does **not** include all the changes!

For full release details, see [here](https://docs.python.org/3/whatsnew/3.9.html)

#### Time Zones

We don't cover 3rd party libraries in this course, but if you've worked with Python in a production environment, you will likely have come across the dreaded timezone and Daylight Savings issues that plague datetimes!

Most likely you will have resorted to using the `pytz` and `python-dateutil` libraries to help with that.

Now, Python 3.9 is proud to introduce the `zoneinfo` module to deal with timezones properly. About time too!

For full info on this, refer to [PEP 615](https://peps.python.org/pep-0615/).

And the Python [docs](https://docs.python.org/3.9/library/zoneinfo.html#module-zoneinfo).

**Windows Users**: you will likely need to add a dependency on the `tzdata` [library](https://pypi.org/project/tzdata/) for the IANA time zone database. See [this note](https://docs.python.org/3.9/library/zoneinfo.html#data-sources)

You should also take a look at this [presentation](https://pganssle-talks.github.io/chipy-nov-2020-zoneinfo/#/) by Paul Ganssle who wrote that module - very interesting read!

Let's look at how we might have handled timezone and DST using `pytz` and `dateutil`, and contrast that to how we can use the new `zoneinfo` module instead.

In [1]:
import zoneinfo
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

import dateutil
import pytz

Let's list out all the defined time zones:

In [2]:
for tz in pytz.all_timezones:
    print(tz)

Africa/Abidjan
Africa/Accra
Africa/Addis_Ababa
Africa/Algiers
Africa/Asmara
Africa/Asmera
Africa/Bamako
Africa/Bangui
Africa/Banjul
Africa/Bissau
Africa/Blantyre
Africa/Brazzaville
Africa/Bujumbura
Africa/Cairo
Africa/Casablanca
Africa/Ceuta
Africa/Conakry
Africa/Dakar
Africa/Dar_es_Salaam
Africa/Djibouti
Africa/Douala
Africa/El_Aaiun
Africa/Freetown
Africa/Gaborone
Africa/Harare
Africa/Johannesburg
Africa/Juba
Africa/Kampala
Africa/Khartoum
Africa/Kigali
Africa/Kinshasa
Africa/Lagos
Africa/Libreville
Africa/Lome
Africa/Luanda
Africa/Lubumbashi
Africa/Lusaka
Africa/Malabo
Africa/Maputo
Africa/Maseru
Africa/Mbabane
Africa/Mogadishu
Africa/Monrovia
Africa/Nairobi
Africa/Ndjamena
Africa/Niamey
Africa/Nouakchott
Africa/Ouagadougou
Africa/Porto-Novo
Africa/Sao_Tome
Africa/Timbuktu
Africa/Tripoli
Africa/Tunis
Africa/Windhoek
America/Adak
America/Anchorage
America/Anguilla
America/Antigua
America/Araguaina
America/Argentina/Buenos_Aires
America/Argentina/Catamarca
America/Argentina/ComodRivad

With the `zoneinfo` module:

In [3]:
for tz in sorted(zoneinfo.available_timezones()):
    print(tz)

Are the time zones defined by `pytz` and `zoneinfo` the same? Yes!

In this example, let's take our current time in UTC, and convert it to some other time zone, say `Australia/Melbourne`.

In [4]:
now_utc_naive = datetime.utcnow()

C:\Users\user1\AppData\Local\Temp\ipykernel_3116\818944004.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now_utc_naive = datetime.utcnow()


In [5]:
now_utc_naive

datetime.datetime(2026, 5, 31, 14, 43, 17, 647891)

The problem here is that we have a _naive_ datetime (i.e. one without an attached timezone).

We can make this naive datetime time zone aware by tacking on the timezone (since we know it is UTC):

In [6]:
now_utc_aware = now_utc_naive.replace(tzinfo=timezone.utc)
now_utc_aware

datetime.datetime(2026, 5, 31, 14, 43, 17, 647891, tzinfo=datetime.timezone.utc)

Or, we could use the `pytz` library to do the same thing:

In [7]:
pytz.utc.localize(datetime.utcnow())

C:\Users\user1\AppData\Local\Temp\ipykernel_3116\1604904123.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  pytz.utc.localize(datetime.utcnow())


datetime.datetime(2026, 5, 31, 14, 43, 19, 203449, tzinfo=<UTC>)

Now that we have a time zone aware datetime, we can convert it to another timezone using `pytz`:

First, let's pick a time zone from `pytz`:

In [8]:
tz_melbourne = pytz.timezone('Australia/Melbourne')

And now we localize our aware datetime to this time zone:

In [9]:
now_utc_aware.astimezone(tz_melbourne)

datetime.datetime(2026, 6, 1, 0, 43, 17, 647891, tzinfo=<DstTzInfo 'Australia/Melbourne' AEST+10:00:00 STD>)

We could do both these steps in a single expression:

In [10]:
now_utc_aware.astimezone(pytz.timezone('Australia/Melbourne'))

datetime.datetime(2026, 6, 1, 0, 43, 17, 647891, tzinfo=<DstTzInfo 'Australia/Melbourne' AEST+10:00:00 STD>)

Now, let's do the same thing using the `zoneinfo` module.

Let's pick the same target time zone:

In [11]:
tz_zi_dublin = ZoneInfo("Europe/Dublin")

ZoneInfoNotFoundError: 'No time zone found with key Europe/Dublin'

And the let's convert our aware datetime to that time zone:

In [ ]:
now_utc_aware.astimezone(tz_zi_dublin)

Or, we can also write this as a single expression:

In [ ]:
now_utc_aware.astimezone(ZoneInfo("Europe/Dublin"))

#### The `math` Module

Several enhancements or additions have been to the math library.

The `math` module already had the `gcd` function to calculate the great common divisor of two numbers:

In [12]:
import math

In [13]:
math.gcd(27, 45)

9

But now `gcd` can take multiple arguments, not just two:

In [14]:
math.gcd(27, 45, 18, 15)

3

The `lcm` (least common multiple) function has been added:

In [15]:
math.lcm(2, 3, 4)

12

#### Dictionary Unions

When we discussed dictionaries in this course, we saw that we could combine two dictionaries using unpacking:

In [16]:
d1 = {'a': 1, 'b': 2, 'c': 3}
d2 = {'c': 30, 'd': 40}

In [17]:
{**d1, **d2}

{'a': 1, 'b': 2, 'c': 30, 'd': 40}

As we saw the second dictionary's key/value pair "overwrote" the key/value pair from the first dictionary.

We could also use the `ChainMap` function in the `collections` module:

In [18]:
from collections import ChainMap

In [19]:
merged = ChainMap(d1, d2)

In [20]:
merged['a'], merged['c'], merged['d']

(1, 3, 40)

As you can see, in the `ChainMap`, the firest occurrence of the key is used - so in this case `c` comes from `d1`, not `d2`.

Both of these ways of "combining" dictionaries work well - but they are not very intuitive, and need a little attention to what happens when you have common keys in the dictionaries.

Think of concatenating lists where we can simply use the `+` operator - this is very intuitive:

In [21]:
[1, 2, 3] + [4, 5, 6]

[1, 2, 3, 4, 5, 6]

Now dictionaries are not like lists, but they are closely related to **sets**. With sets, we have the **union** operator (`|`):

In [22]:
s1 = {'a', 'b', 'c'}
s2 = {'c', 'd'}

s1 | s2

{'a', 'b', 'c', 'd'}

Python 3.9 introduces support for the **union** (`|`) operation between dictionaries as well.

In [23]:
d1 | d2

{'a': 1, 'b': 2, 'c': 30, 'd': 40}

Just like with the `{**d1, **d2}` approach, the value for `c` came from the second dictionary.

And just like with that technique we can control this by switching the order of the dictionaries in the union:

In [24]:
d2 | d1

{'c': 3, 'd': 40, 'a': 1, 'b': 2}

One question we should have, is what happens to the insertion order that Python dictionaries now guarantee?

In [25]:
d1 = {'c': 3, 'a': 1, 'b': 2}
d2 = {'d': 40, 'c': 30}

In [26]:
d1 | d2

{'c': 30, 'a': 1, 'b': 2, 'd': 40}

As you can see, even though the **value** for `c` came from the **second** dictionary, the original inertion order of the **keys** is maintained, so `c` is still in first position in the union of the two dictionaries.

#### String Methods

Often we need to remove some prefix or suffix in a string.

For example, we may have this list of string:

In [27]:
data = [
    "(log) [2022-03-01T13:30:01] Log record 1",
    "(log) [2022-03-01T13:30:02] Log record 2",
    "(log) [2022-03-01T13:30:03] Log record 3",
    "(log) [2022-03-01T13:30:04] Log record 4",
]

And we want to clean these up and remove the `(log) ` prefix (including the space).

We can certainly do it this way:

In [28]:
clean = [
    s.replace("(log) ", '')
    for s in data
]
clean

['[2022-03-01T13:30:01] Log record 1',
 '[2022-03-01T13:30:02] Log record 2',
 '[2022-03-01T13:30:03] Log record 3',
 '[2022-03-01T13:30:04] Log record 4']

You might be tempted to use the `lstrip` method:

In [29]:
clean = [
    s.lstrip("(log) ")
    for s in data
]
clean

['[2022-03-01T13:30:01] Log record 1',
 '[2022-03-01T13:30:02] Log record 2',
 '[2022-03-01T13:30:03] Log record 3',
 '[2022-03-01T13:30:04] Log record 4']

This appears to work, but `lstrip` (and `rstrip`) does not interpet `"(log )"` as a string, but rather a **sequence** of characters, and each one will be removed - so you might end up with this problem:

In [30]:
data2 = [
    "(log) log: [2022-03-01T13:30:01] Log record 1",
    "(log) log: [2022-03-01T13:30:02] Log record 2",
    "(log) log: [2022-03-01T13:30:03] Log record 3",
    "(log) log: [2022-03-01T13:30:04] Log record 4",
]

In [31]:
clean = [
    s.lstrip("(log) ")
    for s in data2
]
clean

[': [2022-03-01T13:30:01] Log record 1',
 ': [2022-03-01T13:30:02] Log record 2',
 ': [2022-03-01T13:30:03] Log record 3',
 ': [2022-03-01T13:30:04] Log record 4']

Now that removed a lot more than expected everything from those strings, unlike the replace, which will replace only the first occurrence by default:

In [32]:
clean = [
    s.replace("(log) ", '')
    for s in data2
]
clean

['log: [2022-03-01T13:30:01] Log record 1',
 'log: [2022-03-01T13:30:02] Log record 2',
 'log: [2022-03-01T13:30:03] Log record 3',
 'log: [2022-03-01T13:30:04] Log record 4']

Python 3,9 introduces two new string methods to do this without having to use `replace`, namely the `removeprefix()` and `removesuffix()` methods:

In [33]:
[
    s.removeprefix("(log) ")
    for s in data
]

['[2022-03-01T13:30:01] Log record 1',
 '[2022-03-01T13:30:02] Log record 2',
 '[2022-03-01T13:30:03] Log record 3',
 '[2022-03-01T13:30:04] Log record 4']

In [34]:
[
    s.removeprefix("(log) ")
    for s in data2
]

['log: [2022-03-01T13:30:01] Log record 1',
 'log: [2022-03-01T13:30:02] Log record 2',
 'log: [2022-03-01T13:30:03] Log record 3',
 'log: [2022-03-01T13:30:04] Log record 4']

Note that if the prefix (or suffix) is not found, nothing happens, the new string will be the same as the original (i.e. no exception is raised):

In [35]:
'Python rocks!'.removeprefix('Java')

'Python rocks!'